In [ ]:
#Importing Data set
import pandas as pd
import numpy as np

sheet_id = "1pvffljfFZwmeeR2BrMdlDYoPOCZt3fqWN-jQLUPBY2M"

# Reading data_set tab
df = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet=data_set")

# Reading the data_dictionary
dd = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet=data_dictionary", header=None)

print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")

Rows: 4358, Columns: 48
Rows: 4358, Columns: 48


In [ ]:
#Looping through every row to understand the data dictionary better
for _, row in dd.iterrows():
    if pd.notna(row[0]) and pd.notna(row[1]) and row[0] != "Field":
        print(f"{row[0]:<40} → {row[1]}")

company_id                               → A unique identifier for this company
name                                     → The name of the company
website                                  → The company's website
phone_number                             → The company's phone number
address                                  → The company's address
office_type                              → The type of office found at this address
claimed_profile                          → Did they reach out to us to claim their profile and provide us with additional information?
is_active                                → Does the company seem to be actively in business?
parent_company_id                        → If the company has a parent, what is that company's parent?
is_cash_home_buyer                       → Is the company considered a cash home buyer company?
company_logo                             → The location of the company's logo on our image server
coverage_type                            → A

In [ ]:
# Data frame for Companies that are active/inactive
print(df["is_active"].value_counts())

# Data frame for Cash home buyers
print(df["is_cash_home_buyer"].value_counts())

#Data frame for coverage type
print(df["coverage_type"].value_counts())

#Data frome for Top 15 states with most companies
print(df["states"].value_counts().head(15))

is_active
True     3747
False     611
Name: count, dtype: int64
is_cash_home_buyer
True     4336
False      19
Name: count, dtype: int64
coverage_type
Local         705
Statewide     163
Nationwide      8
Name: count, dtype: int64
states
Texas             461
Florida           370
California        266
North Carolina    188
Virginia          157
Ohio              137
Pennsylvania      134
Georgia           132
Missouri          122
Tennessee         121
Arizona           117
New Jersey        105
Indiana           104
Michigan           98
Maryland           96
Name: count, dtype: int64


In [ ]:
# Analyzing missing data
# Finding percentage of rows that are empty
# Used a for loop to loop through the data
# .isna() to check for blanks .mean() to calculate percentage
print("=== MISSING DATA ===")
for col in df.columns:
    pct = df[col].isna().mean() * 100  # .isna() to check for blanks, .mean() gives percentage
    if pct > 0:
        print(f"  {col:<40}: {pct:.1f}% missing")

=== MISSING DATA ===
  website                                 : 1.9% missing
  phone_number                            : 1.5% missing
  address                                 : 27.0% missing
  office_type                             : 55.1% missing
  claimed_profile                         : 0.1% missing
  parent_company_id                       : 1.7% missing
  is_cash_home_buyer                      : 0.1% missing
  company_logo                            : 3.2% missing
  coverage_type                           : 79.9% missing
  location_tag                            : 61.8% missing
  num_reviews                             : 33.7% missing
  avg_review_rating                       : 33.7% missing
  reviews_last_6_months                   : 33.7% missing
  avg_rating_last_6_months                : 68.1% missing
  reviews_last_18_months                  : 33.7% missing
  avg_rating_last_18_months               : 53.7% missing
  num_months_with_reviews                 : 33.7% missing

In [ ]:
# Checking how many IDs show up more than one time
count_id = df["company_id"].value_counts() #Count IDs
multi_ids = count_id[count_id > 1] # filtering to only those appearing more than once
print(f"Company IDs appearing more than once: {len(multi_ids)}")


# Checking Franchise structure
# If parent_company_id is different from company_id, then it should be a franchise branch
children = df[df["parent_company_id"] != df["company_id"]]
standalone = df[df["parent_company_id"] == df["company_id"]]
print(f"Franchise branches: {len(children)}")
print(f"Standalone companies: {len(standalone)}")

# We use Opendoor as an example
open_door = df[df["name"] == "Opendoor"]
print(f"\nOpendoor: {len(open_door)}")

Company IDs appearing more than once: 270
Franchise branches: 452
Standalone companies: 3906

Opendoor: 26


In [ ]:
# Creating a copy to not affect the original
df_clean = df.copy()

# Removing the inactive companies since they're not active they can't be recommended
df_clean = df_clean[df_clean["is_active"] == True]

# Removing the non-cash-buyers since we won't be using them in our articles
df_clean = df_clean[df_clean["is_cash_home_buyer"] == 1.0]

# Converting BBB letter grades to numbers
# We need numbers so we can do math with them in scoring
# "Not Rated" becomes NaN (missing)
# Companies that never applied for BBB won't directly be considered as F as this would be unfair
bbb_to_number = {
    "A+": 4.3, "A": 4.0, "A-": 3.7,
    "B+": 3.3, "B": 3.0, "B-": 2.7,
    "C+": 2.3, "C": 2.0, "C-": 1.7,
    "D+": 1.3, "D": 1.0, "D-": 0.7,
    "F": 0.0,
    "Not Rated": np.nan
}
df_clean["bbb_rating_numeric"] = df_clean["bbb_rating"].map(bbb_to_number)

print(f"Rows after cleaning: {len(df_clean)}")

Rows after cleaning: 3729


In [ ]:
# Checking the cities in the DFW metro area
# We use the '|'
# So it matches any company whose cities column contains any of these names
dfw_cities = (
    "Dallas|Fort Worth|Arlington|Plano|Irving|Garland|Grapevine|"
    "Frisco|McKinney|Denton|Mesquite|Grand Prairie|Carrollton|"
    "Richardson|Lewisville|Allen|Flower Mound|Euless|Bedford|"
    "Hurst|Keller|Southlake|Mansfield|Cedar Hill|DeSoto|Rowlett|"
    "Burleson|Wylie|Duncanville"
)

# PATH 1: Local companies with a DFW city are in Texas
is_local = (
    df_clean["cities"].str.contains(dfw_cities, na=False) &
    (df_clean["states"] == "Texas")
)

# PATH 2: Statewide Texas companies
is_statewide = (
    (df_clean["states"] == "Texas") &
    (df_clean["coverage_type"] == "Statewide")
)

# PATH 3: Nationwide companies
is_nationwide = df_clean["coverage_type"] == "Nationwide"

# Combining the 3 paths
market = df_clean[is_local | is_statewide | is_nationwide].copy()
print(f"DFW companies before dedup: {len(market)}")

# We need to remove all duplicate company IDs
# The same company could match multiple times
market = market.drop_duplicates(subset="company_id", keep="first")
print(f"DFW companies after dedup: {len(market)}")

DFW companies before dedup: 160
DFW companies after dedup: 151


In [ ]:
# Test 1: CREDIBILITY (This would be 40% of our overall score)
def score_credibility(row):
    score = 0                # We start with 0 points on the scale#

    # BBB Rating: (0 - 30)ponits  we will score this between 0 - 30 points
    # A+ will have full 30 points, F will have 0 points
    if pd.notna(row.get("bbb_rating_numeric")):
        score += (row["bbb_rating_numeric"] / 4.3) * 30
    else:
        score += 12
    # If no BBB data available we will give 12 out of 30

    # BBB Accreditation: (0 - 10) points
    # Accredited (paid for and earned BBB membership)
    if row.get("bbb_accreditation") == 1.0:
        score += 10
    elif pd.isna(row.get("bbb_accreditation")):
        score += 4  # no data assumed neutral since it would be unfair as not 0

    # Review Volume: (0 - 25) points
    # More reviews assumed to have more social presence
    # We use the log scale here so values are under the limit to evaluate better
    # Log scale would normalize some of the high/low values to even the scoring
    reviews = row.get("num_reviews", 0) or 0
    if reviews > 0:
        score += min(25, np.log1p(reviews) / np.log1p(200) * 25)

    # Years in Business: (0 - 20) points
    # Longer years in business would account for more trust
    # We can have a cap of 7 years saying any company with 7 or more years
    months = row.get("total_months_active", 0) or 0
    years = months / 12
    score += min(20, years * (20 / 7))

    # Claimed Profile: (0 - 10) points
    # If the company actively claimed their profile online presence awareness
    if row.get("claimed_profile") == 1.0:
        score += 10

    # Website + Phone: (0 - 5) points
    # Legitimacy of real businesses having a website and phone number
    if pd.notna(row.get("website")):
        score += 2.5
    if pd.notna(row.get("phone_number")):
        score += 2.5

    # Cap at 100 (some companies might exceed with all maximums)
    return min(100, score)


# Test 2: CUSTOMER EXPERIENCE (This would be 35% of final score)
def score_experience(row):
    score = 0
    # Checking if the company has reviews
    has_reviews = pd.notna(row.get("num_reviews")) and row["num_reviews"] > 0

    if has_reviews:
        # Average Review Rating: (0 - 35) points
        # A 5.0 gets 35 points, a 3.0 gets 10, below 3.0 gets very little
        # Taking into account an acceptable range of 3.0 and more
        # Considering a score of 10 for 3.0
        rating = row.get("avg_review_rating", 0) or 0
        if rating >= 3.0:
            score += 10 + (rating - 3.0) / 2.0 * 25
        else:
            score += max(0, (rating - 1.0) / 2.0 * 10)

        # Google Review Rating: 0 to 15 points
        # If they have 4.9 on Trustpilot but 1.4 on Google, something's wrong
        g_rating = row.get("google_avg_review_rating")
        if pd.notna(g_rating):
            if g_rating >= 3.0:
                score += 5 + (g_rating - 3.0) / 2.0 * 10
            else:
                score += max(0, (g_rating - 1.0) / 2.0 * 5)
        else:
            score += 7.5  # neutral value for no Google data

        # 1-Star Review Penalty: (up to -15 points)
        # A company might have a 4.5 average but 20% of reviews are 1-star
        # That means 1 in 5 customers had a terrible experience
        # we make a penalty if more than 15% of reviews are 1 star
        total = row.get("num_reviews", 1) or 1
        one_star = row.get("num_1_star_reviews", 0) or 0
        one_star_pct = one_star / total
        if one_star_pct > 0.15:
            score -= min(15, (one_star_pct - 0.15) * 75)

        # Response Rate: (0 - 15) points
        # Companies that reply to reviews are customer and reputation friendly
        # 100% response rate = 15 points, 0% = 0 points
        resp = row.get("response_rate", 0) or 0
        score += resp * 15

        # BBB Complaints Penalty: (up to -10 points)
        # Formal complaints are a little worse than bad reviews as red flags
        # Assuming each complaint costs 3 points and max penalty is 10
        complaints = row.get("bbb_complaints", 0) or 0
        if complaints > 0:
            score -= min(10, complaints * 3)
    else:
        # Having no reviews #
        # We can't judge their experience, so give a neutral 30 out of 100
        # we can't completely ignore them either as could be good companies
        # We can't give them a higher ranking since no evidence
        score = 30

    # Keep score between 0 and 100 ensuring that values don't exceed the limit
    return max(0, min(100, score))


# Test 3: LOCAL ACTIVITY (25% of final score)
def score_local_activity(row):
    score = 0
    has_reviews = pd.notna(row.get("num_reviews")) and row["num_reviews"] > 0

    if has_reviews:
        #  Reviews in Last 6 Months: (0 to 35 points)
        # This is the strongest active right now signal
        # If they got reviews recently, thats a good sign
        # Log scale again recent review matters a lot, 100 vs 200 matters less
        r6 = row.get("reviews_last_6_months", 0) or 0
        score += min(35, np.log1p(r6) / np.log1p(10) * 35)

        # Reviews in Last 18 Months: (0 to 20) points
        # Wider window — maybe they had a slow quarter but are generally active
        r18 = row.get("reviews_last_18_months", 0) or 0
        score += min(20, np.log1p(r18) / np.log1p(20) * 20)

        #  Months Since Last Review: (0 to 20) points
        # If their last review was 3 years ago, they're probably not active here
        mslr = row.get("months_since_last_review", 999) or 999
        if mslr <= 1:      # reviewed this month or last month
            score += 20
        elif mslr <= 3:    # within last quarter
            score += 15
        elif mslr <= 6:    # within last half year
            score += 10
        elif mslr <= 12:   # within last year
            score += 5
        # Over 12 months ago = 0 points here

        # Review Consistency: (0 to 10) points
        # Getting reviews every month would count as steady business
        # vs. getting 50 reviews in one month then nothing for 2 years
        pct = row.get("pct_months_with_review", 0) or 0
        score += pct * 10
    else:
        # No reviews would say that very little evidence they're active here
        score = 15

    # Coverage Type Bonus: (0 to 15) points
    # A local company with a DFW office is almost certainly buying in DFW
    # A nationwide company listed in 30 markets might not be active in all
    coverage = row.get("coverage_type")
    if coverage == "Local":
        score += 15        # highest bonus — they're focused here
    elif coverage == "Statewide":
        score += 10        # pretty good — Texas-focused
    elif coverage == "Nationwide":
        score += 5         # lowest — spread across many markets
    else:
        score += 10        # unknown coverage

    return max(0, min(100, score))

print("All three scoring functions defined successfully")

All three scoring functions defined successfully


In [ ]:
# Running each scoring function on every row
# .apply()
market["credibility_score"] = market.apply(score_credibility, axis=1)
market["experience_score"] = market.apply(score_experience, axis=1)
market["local_activity_score"] = market.apply(score_local_activity, axis=1)

# Combine into one composite score using the weights
# Credibility is most important (40%), then Experience (35%), Activity (25%)
market["composite_score"] = (
    market["credibility_score"] * 0.40 +
    market["experience_score"] * 0.35 +
    market["local_activity_score"] * 0.25
)

# Sort from highest score to lowest
market = market.sort_values("composite_score", ascending=False)

# Assign rank numbers: 1, 2, 3..
market["rank"] = range(1, len(market) + 1)

# Print the top 15 as a table
print("=" * 85)
print(f"{'Rank':<5} {'Company':<30} {'Score':>6} {'Cred':>6} {'Exp':>6} {'Active':>7}")
print("=" * 85)
for _, r in market.head(15).iterrows():
    print(f"{int(r['rank']):<5} {r['name'][:29]:<30} {r['composite_score']:>6.1f} "
          f"{r['credibility_score']:>6.1f} {r['experience_score']:>6.1f} "
          f"{r['local_activity_score']:>7.1f}")

Rank  Company                         Score   Cred    Exp  Active
1     Clever Offers                    78.5  100.0   60.2    69.9
2     TX Cash Home Buyers              77.0   91.6   58.3    79.9
3     Greenlight Offer                 74.6   95.2   52.6    72.6
4     Big State Home Buyers            74.6   90.0   50.0    84.5
5     Ninebird Properties              71.9   85.7   62.6    62.9
6     Texas Best                       70.5   77.2   61.0    72.9
7     Metroplex Homebuyers             70.0   88.7   55.2    60.7
8     TX Home Buying Pros              69.6   76.7   60.3    71.2
9     Starfish Group Properties        69.4   72.4   64.1    71.8
10    Orchard                          68.4   88.3   40.8    75.0
11    Dioro Home Buyers                67.1   67.4   60.9    75.3
12    Trusted Home Offer               65.6   71.8   61.3    61.9
13    Duke Real Estate & Asset Mana    65.5   76.1   44.9    77.3
14    Home Buying Guys                 65.3   75.8   62.2    52.7
15    Hous

In [ ]:
# For each of the top 10, we can print all the key metrics
for _, r in market.head(10).iterrows():
    print(f"\n#{int(r['rank'])} — {r['name']}")
    print(f"   Composite Score: {r['composite_score']:.1f}")
    print(f"   Credibility: {r['credibility_score']:.1f} | Experience: {r['experience_score']:.1f} | Activity: {r['local_activity_score']:.1f}")
    print(f"   Reviews: {r['num_reviews']}, Avg Rating: {r['avg_review_rating']}")
    print(f"   Reviews last 6 months: {r['reviews_last_6_months']}")
    print(f"   Google Rating: {r['google_avg_review_rating']}")
    print(f"   BBB Rating: {r['bbb_rating']}, Active since: {r['year_first_active']}")
    print(f"   Coverage: {r['coverage_type']}")


#1 — Clever Offers
   Composite Score: 78.5
   Credibility: 100.0 | Experience: 60.2 | Activity: 69.9
   Reviews: 4294.0, Avg Rating: 4.916627853
   Reviews last 6 months: 240.0
   Google Rating: 4.763747454
   BBB Rating: A+, Active since: 2018.0
   Coverage: Nationwide

#2 — TX Cash Home Buyers
   Composite Score: 77.0
   Credibility: 91.6 | Experience: 58.3 | Activity: 79.9
   Reviews: 47.0, Avg Rating: 4.723404255
   Reviews last 6 months: 6.0
   Google Rating: 4.790697674
   BBB Rating: A+, Active since: 2019.0
   Coverage: Statewide

#3 — Greenlight Offer
   Composite Score: 74.6
   Credibility: 95.2 | Experience: 52.6 | Activity: 72.6
   Reviews: 161.0, Avg Rating: 4.50931677
   Reviews last 6 months: 29.0
   Google Rating: 4.525316456
   BBB Rating: A+, Active since: 2020.0
   Coverage: Statewide

#4 — Big State Home Buyers
   Composite Score: 74.6
   Credibility: 90.0 | Experience: 50.0 | Activity: 84.5
   Reviews: 224.0, Avg Rating: 4.709821429
   Reviews last 6 months: 18.0

In [ ]:
# Finding Opendoor in our market
od = market[market["name"] == "Opendoor"]
if len(od) > 0:
    r = od.iloc[0]
    print(f"Opendoor ranks #{int(r['rank'])} out of {len(market)} companies")
    print(f"   Composite Score: {r['composite_score']:.1f}")
    print(f"   Credibility: {r['credibility_score']:.1f} (HIGH — big brand, lots of reviews)")
    print(f"   Experience: {r['experience_score']:.1f} (LOW — customers are unhappy)")
    print(f"   Activity: {r['local_activity_score']:.1f}")
    print(f"   Avg Rating: {r['avg_review_rating']:.2f}")
    print(f"   Google Rating: {r['google_avg_review_rating']}")
    print(f"   1-star reviews: {r['num_1_star_reviews']} out of {r['num_reviews']} total")
    one_star_pct = r['num_1_star_reviews'] / r['num_reviews'] * 100
    print(f"   That's {one_star_pct:.0f}% one-star reviews")
else:
    print("Opendoor not found in DFW market")



Opendoor ranks #27 out of 151 companies
   Composite Score: 60.2
   Credibility: 90.0 (HIGH — big brand, lots of reviews)
   Experience: 18.7 (LOW — customers are unhappy)
   Activity: 70.6
   Avg Rating: 3.72
   Google Rating: 1.4
   1-star reviews: 512.0 out of 1716.0 total
   That's 30% one-star reviews


In [ ]:
# Bottom 10 should have clear weaknesses
print("BOTTOM 10")
for _, r in market.tail(10).iterrows():
    print(f"\n#{int(r['rank'])} — {r['name']}")
    print(f"   Score: {r['composite_score']:.1f}")
    print(f"   Reviews: {r['num_reviews']}, Rating: {r['avg_review_rating']}")
    print(f"   BBB: {r['bbb_rating']}")
    print(f"   Last review: {r['months_since_last_review']} months ago")

# You should see companies with things like:
# - No reviews or very few reviews
# - No BBB data
# - Last review was years ago
# - Bad ratings
# - BBB rating of F

BOTTOM 10

#142 — Shannon buys houses fast
   Score: 33.1
   Reviews: 1.0, Rating: 5.0
   BBB: nan
   Last review: 34.0 months ago

#143 — Solid Offers
   Score: 32.8
   Reviews: 11.0, Rating: 3.909090909
   BBB: nan
   Last review: 22.0 months ago

#144 — Higher Home Buyer
   Score: 32.6
   Reviews: 1.0, Rating: 5.0
   BBB: nan
   Last review: 28.0 months ago

#145 — Cash Home Buyers Texas - Denton
   Score: 32.4
   Reviews: 1.0, Rating: 5.0
   BBB: nan
   Last review: 14.0 months ago

#146 — JKV Homebuyers
   Score: 32.2
   Reviews: 7.0, Rating: 3.428571429
   BBB: nan
   Last review: 75.0 months ago

#147 — We Buy Houses Dallas .US
   Score: 32.1
   Reviews: nan, Rating: nan
   BBB: nan
   Last review: nan months ago

#148 — We Buy Fire Damaged Houses
   Score: 31.7
   Reviews: 7.0, Rating: 3.857142857
   BBB: nan
   Last review: 0.0 months ago

#149 — Joe Homebuyer Dallas
   Score: 31.6
   Reviews: 5.0, Rating: 4.2
   BBB: Not Rated
   Last review: 42.0 months ago

#150 — Southern 

In [ ]:
print("SCORE DISTRIBUTION")
print(f"Top 10 range:    {market.head(10)['composite_score'].min():.1f} to {market.head(10)['composite_score'].max():.1f}")
print(f"Median:          {market['composite_score'].median():.1f}")
print(f"Bottom 10 range: {market.tail(10)['composite_score'].min():.1f} to {market.tail(10)['composite_score'].max():.1f}")



SCORE DISTRIBUTION
Top 10 range:    68.4 to 78.5
Median:          44.7
Bottom 10 range: 25.9 to 33.1


In [ ]:
# Columns needed for article
output_cols = [
    "rank", "name", "composite_score",
    "credibility_score", "experience_score", "local_activity_score",
    "website", "phone_number",
    "num_reviews", "avg_review_rating", "reviews_last_6_months",
    "google_num_reviews", "google_avg_review_rating",
    "bbb_rating", "bbb_accreditation",
    "year_first_active", "total_months_active",
    "coverage_type", "cities",
    "response_rate", "months_since_last_review",
    "num_1_star_reviews"
]

# Save to CSV file
market[output_cols].to_csv("dfw_ranked_list.csv", index=False)
print("Saved to dfw_ranked_list.csv")

Saved to dfw_ranked_list.csv
